In [ ]:
import os
import time
import random
from pathlib import Path
import tqdm
import numpy as np
import torch
import sklearn
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, recall_score

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)
import datasets
from datasets import Dataset

from src.models.baseline_model import BaselineModel
from src.models.transformer_model import TransformerModel
from src.utils.lyrics_data_processor import LyricsDataProcessor

# Deshabilitar WanDB
os.environ["WANDB_DISABLED"] = "true"

print("- Versiones de librerías utilizadas:")
print(f"    - NumPy: {np.__version__}")
print(f"    - PyTorch: {torch.__version__}")
print(f"    - transformers: {transformers.__version__}")
print(f"    - datasets: {datasets.__version__}")
print(f"    - scikit-learn: {sklearn.__version__}")

- Versiones de librerías utilizadas:
    - NumPy: 2.2.6
    - PyTorch: 2.9.0+cu128
    - transformers: 4.57.1
    - datasets: 4.4.2
    - scikit-learn: 1.8.0


In [ ]:
# Semilla
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"- Semilla utilizada: {SEED}")

- Semilla utilizada: 42


In [ ]:
# Parámetros
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"- Dispositivo para PyTorch: {DEVICE}")

# Directorios
DATA_DIR = Path("data")
os.makedirs(DATA_DIR, exist_ok=True)
GENERATED_DATA_DIR = DATA_DIR / "generated"
os.makedirs(GENERATED_DATA_DIR, exist_ok=True)
DATASETS_TRANSFORMER = {
    "train": Path(DATA_DIR / "train_dataset_transformer"),
    "eval": Path(DATA_DIR / "eval_dataset_transformer"),
    "test": Path(DATA_DIR / "test_dataset_transformer")
}
DATASETS_BASELINE = {
    "train": (Path(DATA_DIR / "X_train_baseline.npz"), Path(DATA_DIR / "y_train.npy")),
    "eval": (Path(DATA_DIR / "X_eval_baseline.npz"), Path(DATA_DIR / "y_eval.npy")),
    "test": (Path(DATA_DIR / "X_test_baseline.npz"), Path(DATA_DIR / "y_test.npy"))
}
EMBEDDINGS = {
    "train": Path(GENERATED_DATA_DIR / "train_embeddings.npy"),
    "eval": Path(GENERATED_DATA_DIR / "eval_embeddings.npy"),
    "test": Path(GENERATED_DATA_DIR / "test_embeddings.npy")
}
MODELS_DIR = Path("models")
os.makedirs(MODELS_DIR, exist_ok=True)
TEACHER_MODEL_PATH = MODELS_DIR / "transformer"  # Modelo Transformer de la práctica 1 (teacher en la práctica 2)
BASELINE_MODEL_PATH = MODELS_DIR / "baseline_model"  # Modelo baseline de la práctica 1
STUDENT_MODEL_PATH = MODELS_DIR / "student_transformer"  # Modelo Transformer student

print(f"- Directorios establecidos:")
print(f"    - DATA_DIR: {str(DATA_DIR)}")
print(f"    - GENERATED_DATA_DIR: {str(GENERATED_DATA_DIR)}")
print(f"    - MODELS_DIR: {str(MODELS_DIR)}")
print(f"        - TEACHER_MODEL: {str(TEACHER_MODEL_PATH)} --> {"Existe" if TEACHER_MODEL_PATH.exists() else "No existe"}")
print(f"        - BASELINE_MODEL: {str(BASELINE_MODEL_PATH)} --> {"Existe" if BASELINE_MODEL_PATH.exists() else "No existe"}")
print(f"        - STUDENT_MODEL: {str(STUDENT_MODEL_PATH)} --> {"Existe" if STUDENT_MODEL_PATH.exists() else "No existe"}")
print(f"    - DATASETS_TRANSFORMER: {dict(map(lambda x: (x, str(DATASETS_TRANSFORMER[x])), DATASETS_TRANSFORMER))}")
print(f"    - DATASETS_BASELINE: {dict(map(lambda x: (x, (str(DATASETS_BASELINE[x][0]), str(DATASETS_BASELINE[x][1]))), DATASETS_BASELINE))}")
print(f"    - EMBEDDINGS: {dict(map(lambda x: (x, str(EMBEDDINGS[x])), EMBEDDINGS))}")

- Dispositivo para PyTorch: cuda
- Directorios establecidos:
    - DATA_DIR: data
    - GENERATED_DATA_DIR: data/generated
    - MODELS_DIR: models
        - TEACHER_MODEL: models/transformer --> Existe
        - BASELINE_MODEL: models/baseline_model --> No existe
        - STUDENT_MODEL: models/student_transformer --> Existe
    - DATASETS_TRANSFORMER: {'train': 'data/train_dataset_transformer', 'eval': 'data/eval_dataset_transformer', 'test': 'data/test_dataset_transformer'}
    - DATASETS_BASELINE: {'train': ('data/X_train_baseline.npz', 'data/y_train.npy'), 'eval': ('data/X_eval_baseline.npz', 'data/y_eval.npy'), 'test': ('data/X_test_baseline.npz', 'data/y_test.npy')}
    - EMBEDDINGS: {'train': 'data/generated/train_embeddings.npy', 'eval': 'data/generated/eval_embeddings.npy', 'test': 'data/generated/test_embeddings.npy'}


In [ ]:
# Comprobamos la existencia de los datos
for split in ["train", "eval", "test"]:
    assert DATASETS_TRANSFORMER[split].exists(), f"El conjunto de datos {split} para el modelo Transformer no existe."
    assert DATASETS_BASELINE[split][0].exists(), f"El conjunto de datos {split} para el modelo baseline no existe."
    assert DATASETS_BASELINE[split][1].exists(), f"El conjunto de etiquetas {split} para el modelo baseline no existe."

In [ ]:
# Comprobamos la existencia de los embeddings generados
for split, path in EMBEDDINGS.items():
    if not path.exists():
        EMBEDDINGS[split] = None
        print(f"- Los embeddings para el conjunto {split} no existen.")
    else:
        print(f"- Los embeddings para el conjunto {split} existen.")

- Los embeddings para el conjunto train existen.
- Los embeddings para el conjunto eval existen.
- Los embeddings para el conjunto test existen.


# Procesamiento del Lenguaje Natural 2 - Práctica 2
## 1. Módulo de *retrieval* denso
### 1.1. Construcción del índice:
- Utilizamos el conjunto de entrenamiento de la práctica 1 como corpus.

In [ ]:
# Datos preparados para el modelo Transformer
train_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(DATA_DIR / "train_dataset_transformer")
eval_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(DATA_DIR / "eval_dataset_transformer")
test_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(DATA_DIR / "test_dataset_transformer")

# Datos preparados para el modelo baseline
X_train_baseline = LyricsDataProcessor.load_data_baseline(DATA_DIR / "X_train_baseline.npz")
y_train_baseline = LyricsDataProcessor.load_label(DATA_DIR / "y_train.npy")
X_eval_baseline = LyricsDataProcessor.load_data_baseline(DATA_DIR / "X_eval_baseline.npz")
y_eval_baseline = LyricsDataProcessor.load_label(DATA_DIR / "y_eval.npy")
X_test_baseline = LyricsDataProcessor.load_data_baseline(DATA_DIR / "X_test_baseline.npz")
y_test_baseline = LyricsDataProcessor.load_label(DATA_DIR / "y_test.npy")

### 1.2. Función de *embedding*: 
- `embed_fn(texts)` usando el *encoder* del modelo Transformer de la práctica 1 (sin la capa de clasificación) para obtener vectores $d$-dimensionales.

In [ ]:
def embed_fn(transformer_model, texts, device):
    embeddings = []
    for text_id in tqdm.tqdm(range(len(texts)), desc="Obteniendo embeddings"):
        input_ids = torch.tensor(texts[text_id]["input_ids"]).to(device)
        attention_mask = torch.tensor(texts[text_id]["attention_mask"]).to(device)

        inputs = {"input_ids": input_ids.unsqueeze(0), "attention_mask": attention_mask.unsqueeze(0)}

        with torch.no_grad():
            outputs = transformer_model._model(**inputs, output_hidden_states=True)
            e = outputs.hidden_states[-1][:, 0, :]

        embeddings.append(e.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
# Cargamos el modelo Transformer preentrenado
LABEL2ID = {
    "rap": 0,
    "rock": 1,
    "pop": 2,
    "other": 3
}
ID2LABEL = {
    0: "rap",
    1: "rock",
    2: "pop",
    3: "other"
}
teacher_model = TransformerModel.load_model(
    path=MODELS_DIR / "transformer",
    label2id=LABEL2ID,
    id2label=ID2LABEL
)

- Cargando modelo desde models/transformer...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


### 1.3. Índice k-NN
- Construir un índice (por ejemplo, con `sklearn.NearestNeighbors` y métrica coseno).

In [ ]:
# Construir índice k-NN con métrica coseno
N_NEIGHBORS = 5
nearest_neighbors = NearestNeighbors(n_neighbors=N_NEIGHBORS, metric="cosine")

# Obtener embeddings del conjunto de entrenamiento
if EMBEDDINGS["train"] is None:
    train_embeddings = embed_fn(teacher_model, train_dataset_transformer, DEVICE)
    np.save(GENERATED_DATA_DIR / "train_embeddings.npy", train_embeddings)
else:
    train_embeddings = np.load(EMBEDDINGS["train"])

# Obtener embeddings del conjunto de evaluación
if EMBEDDINGS["eval"] is None:
    eval_embeddings = embed_fn(teacher_model, eval_dataset_transformer, DEVICE)
    np.save(GENERATED_DATA_DIR / "eval_embeddings.npy", eval_embeddings)
else:
    eval_embeddings = np.load(EMBEDDINGS["eval"])

# Obtener embeddings del conjunto de test
if EMBEDDINGS["test"] is None:
    test_embeddings = embed_fn(teacher_model, test_dataset_transformer, DEVICE)
    np.save(GENERATED_DATA_DIR / "test_embeddings.npy", test_embeddings)
else:
    test_embeddings = np.load(EMBEDDINGS["test"])

In [ ]:
# Ajustar el índice con los embeddings
nearest_neighbors.fit(train_embeddings)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


### 1.4. Búsqueda
- Definir `search(query_texts, k)` para devolver:
    - Los $k$ vecinos más cercanos
    - La distancia/similitud.
    - Metadatos asociados (texto original, etiqueta).


In [ ]:
def search(
        nearest_neighbors: NearestNeighbors,
        transformer_model: TransformerModel,
        knn_dataset: Dataset | dict,
        query_texts: Dataset | dict | None = None,
        embeddings: np.ndarray | None = None,
        k: int = 5,
        device: torch.device = torch.device("cpu")
    ) -> tuple[np.ndarray, np.ndarray, list[str], list[int]]:
    if query_texts is None and embeddings is None:
        raise ValueError("Se debe proporcionar 'query_texts' o 'embeddings'.")

    query_embeddings = embeddings if embeddings is not None else embed_fn(transformer_model, query_texts, device)
    distances, indices = nearest_neighbors.kneighbors(query_embeddings, n_neighbors=k)

    texts = []
    labels = []
    for row_indices in indices:
        texts.append([knn_dataset[i]["text"] for i in row_indices])
        labels.append([knn_dataset[i]["label"] for i in row_indices])

    return distances, indices, texts, labels

### 1.5. Evaluación básica
- Calcular Precision@k y/o Recall@k de los vecinos que tienen la misma etiqueta que el ejemplo objetivo.

In [ ]:
# Obtenemos un ejemplo aleatorio del conjunto de evaluación
example_idx = random.randint(0, len(eval_dataset_transformer) - 1)
example = eval_dataset_transformer[example_idx]

# Realizamos la búsqueda
query_texts = [example]
query_label = example["label"]
distances, indices, texts, labels = search(
    query_texts=query_texts,
    nearest_neighbors=nearest_neighbors,
    k=N_NEIGHBORS,
    transformer_model=teacher_model,
    device=DEVICE,
    knn_dataset=train_dataset_transformer
)

print(f"- Ejemplo de consulta (índice {example_idx} del conjunto de evaluación):")
print(f"    - Texto: {example['text'][:100]}...")
print(f"    - Etiqueta verdadera: {query_label}")
print(f"    - Vecinos más cercanos encontrados:")
for i in range(N_NEIGHBORS):
    print(f"        - Vecino {i+1}:")
    print(f"            - Distancia: {distances[0][i]:.4f}")
    print(f"            - Texto: {texts[0][i][:100]}...")
    print(f"            - Etiqueta: {labels[0][i]}")

Obteniendo embeddings: 100%|█████████████████████| 1/1 [00:00<00:00,  7.73it/s]


- Ejemplo de consulta (índice 41905 del conjunto de evaluación):
    - Texto: Bells ringing inside my head
Now, I could ignore them
But I choose to live instead
So I take up into...
    - Etiqueta verdadera: 2
    - Vecinos más cercanos encontrados:
        - Vecino 1:
            - Distancia: 0.0125
            - Texto: Stripped to the waist
We fall into the river
Cover your eyes
So you don't know the secret
I've been ...
            - Etiqueta: 2
        - Vecino 2:
            - Distancia: 0.0136
            - Texto: I don't wanna walk away
But I can't stay how you say I should be
I won't break
But I can't bend enou...
            - Etiqueta: 2
        - Vecino 3:
            - Distancia: 0.0146
            - Texto: I'm freaking out
Yeah, yeah
I'm standing in the heart of nowhere
Futility's forgotten soldier
The da...
            - Etiqueta: 2
        - Vecino 4:
            - Distancia: 0.0147
            - Texto: Cause I use to live
In a fuzzy dream
And I wanted to be
Like all the

In [ ]:
test_distances, test_indices, test_texts, test_labels = search(
    embeddings=test_embeddings,
    nearest_neighbors=nearest_neighbors,
    k=N_NEIGHBORS,
    transformer_model=teacher_model,
    device=DEVICE,
    knn_dataset=train_dataset_transformer
)

# Precomputar el conteo de ejemplos por clase en el conjunto de entrenamiento
class_counts = {}
for j in range(len(train_dataset_transformer)):
    label = train_dataset_transformer[j]["label"]
    class_counts[label] = class_counts.get(label, 0) + 1

# Calcular Precision@k y Recall@k para cada ejemplo de test
precision_at_k = []
recall_at_k = []

for i, test_label in enumerate(test_labels):
    query_label = y_test_baseline[i]
    
    # Precision@k: fracción de los k vecinos que tienen la etiqueta correcta
    relevant_neighbors = sum(1 for label in test_label if label == query_label)
    prec_k = relevant_neighbors / N_NEIGHBORS
    precision_at_k.append(prec_k)

    # Recall@k: fracción de los ejemplos relevantes encontrados en los k vecinos
    total_relevant_in_train = class_counts.get(query_label, 0)
    rec_k = relevant_neighbors / total_relevant_in_train if total_relevant_in_train > 0 else 0
    recall_at_k.append(rec_k)

# Calcular promedios
avg_precision_at_k = np.mean(precision_at_k)
avg_recall_at_k = np.mean(recall_at_k)

print(f"- Precision@{N_NEIGHBORS}: {avg_precision_at_k:.4f}")
print(f"- Recall@{N_NEIGHBORS}: {avg_recall_at_k:.4f}")
print(f"- Número de ejemplos de test evaluados: {len(test_labels)}")

- Precision@5: 0.7834
- Recall@5: 0.0001
- Número de ejemplos de test evaluados: 47638


## 2. Clasificador k-NN sobre *embeddings*
### 2.1. Definición del clasificador
- Implementar un clasificador que obtenga el embedding del texto, recupere los top−k vecinos de entrenamiento y prediga la clase mediante voto mayoritario de las etiquetas de los vecinos.

In [ ]:
class KNNClassifier:
    def __init__(
            self,
            transformer_model: TransformerModel,
            device: torch.device,
            k: int
        ):
        """
        Constructor de la clase KNNClassifier

        :param transformer_model: Transformer para obtener embeddings.
        :type transformer_model: TransformerModel
        :param device: Dispositivo para PyTorch.
        :type device: torch.device
        :param k: Número de vecinos a considerar.
        :type k: int
        """
        self.transformer_model = transformer_model
        self.device = device
        self.k = k
        self.nearest_neighbors = None

    def fit(self, train_dataset: dict | Dataset, train_embeddings: np.ndarray | None = None):
        """
        Entrena el clasificador k-NN con el conjunto de entrenamiento dado.
        :param train_dataset: Conjunto de entrenamiento.
        :type train_dataset: dict | Dataset
        """
        self.train_dataset = train_dataset
        self.train_embeddings = train_embeddings if train_embeddings is not None else self._embed_fn(train_dataset)
        self.nearest_neighbors = NearestNeighbors(n_neighbors=self.k)
        self.nearest_neighbors.fit(self.train_embeddings)

    def predict(self, query_dataset: dict | Dataset | None = None, embeddings: np.ndarray | None = None):
        if query_dataset is None and embeddings is None:
            raise ValueError("Se debe proporcionar un conjunto de datos o embeddings para la predicción.")
        embs = embeddings if embeddings is not None else self._embed_fn(query_dataset)
        _, indices, _, _ = self._search(embs)
        predictions = []
        for neighbor_indices in indices:
            neighbor_labels = [self.train_dataset[i]["label"] for i in neighbor_indices]
            pred_label = max(set(neighbor_labels), key=neighbor_labels.count)
            predictions.append(pred_label)
        return indices, np.array(predictions)
    
    def _embed_fn(self, texts: Dataset | dict):
        embeddings = []
        for text_id in tqdm.tqdm(range(len(texts)), desc="Obteniendo embeddings"):
            input_ids = torch.tensor(texts[text_id]["input_ids"]).to(self.device)
            attention_mask = torch.tensor(texts[text_id]["attention_mask"]).to(self.device)

            inputs = {"input_ids": input_ids.unsqueeze(0), "attention_mask": attention_mask.unsqueeze(0)}

            with torch.no_grad():
                outputs = self.transformer_model._model(**inputs, output_hidden_states=True)
                e = outputs.hidden_states[-1][:, 0, :]
            
            embeddings.append(e.cpu().numpy())
        
        return np.vstack(embeddings)
    
    def _search(self, query_embeddings: np.ndarray | None = None):

        distances, indices = self.nearest_neighbors.kneighbors(query_embeddings, n_neighbors=self.k)
        
        texts = []
        labels = []
        for row_indices in indices:
            texts.append([self.train_dataset[i]["text"] for i in row_indices])
            labels.append([self.train_dataset[i]["label"] for i in row_indices])
        
        return distances, indices, texts, labels

### 2.2. Evaluación
- Medir el rendimiento (Accuracy y Macro-F1) usando la misma partición de datos (train/dev/test) que en la práctica 1.

In [ ]:
knn_classifier = KNNClassifier(teacher_model, DEVICE, N_NEIGHBORS)
knn_classifier.fit(train_dataset_transformer, train_embeddings)
knn_eval_indices, knn_eval_predictions = knn_classifier.predict(embeddings=eval_embeddings)

accuracy_eval_knn = accuracy_score(y_eval_baseline, knn_eval_predictions)
macrof1_eval_knn = f1_score(y_eval_baseline, knn_eval_predictions, average="macro")
perclassf1_eval_knn = f1_score(y_eval_baseline, knn_eval_predictions, average=None)
print("- Resultados del clasificador k-NN con el conjunto de evaluación:")
print(f"    - Accuracy: {accuracy_eval_knn:.3f}")
print(f"    - Macro-F1: {macrof1_eval_knn:.3f}")
print("    - Per-class F1:")
for i, label in ID2LABEL.items():
    print(f"        - {label}: {perclassf1_eval_knn[i]:.3f}")

- Resultados del clasificador k-NN con el conjunto de evaluación:
    - Accuracy: 0.802
    - Macro-F1: 0.802
    - Per-class F1:
        - rap: 0.948
        - rock: 0.714
        - pop: 0.756
        - other: 0.790


In [ ]:
# Registro centralizado para test (para la sección 7)
PRED_TEST = {}
METRICS_TEST = {}

In [ ]:
# Evaluamos el clasificador k-NN en el conjunto de test
knn_test_indices, knn_test_predictions = knn_classifier.predict(embeddings=test_embeddings)
accuracy_test_knn = accuracy_score(y_test_baseline, knn_test_predictions)
macrof1_test_knn = f1_score(y_test_baseline, knn_test_predictions, average="macro")
perclassf1_test_knn = f1_score(y_test_baseline, knn_test_predictions, average=None)

PRED_TEST["knn_embeddings"] = knn_test_predictions
METRICS_TEST["knn_embeddings"] = {
    "accuracy": accuracy_test_knn,
    "macrof1": macrof1_test_knn,
    "perclassf1": perclassf1_test_knn
}


print("- Resultados del clasificador k-NN con el conjunto de test:")
print(f"    - Accuracy: {accuracy_test_knn:.3f}")
print(f"    - Macro-F1: {macrof1_test_knn:.3f}")
print("    - Per-class F1:")
for i, label in ID2LABEL.items():
    print(f"        - {label}: {perclassf1_test_knn[i]:.3f}")

- Resultados del clasificador k-NN con el conjunto de test:
    - Accuracy: 0.806
    - Macro-F1: 0.806
    - Per-class F1:
        - rap: 0.950
        - rock: 0.721
        - pop: 0.764
        - other: 0.790


### 2.3. Comparación
- Comparar el rendimiento del k-NN con el modelo clásico (TF-IDF + clasificador de P1) y el Transformer fine-tuneado en la práctica 1.

In [ ]:
# Cargamos el modelo clásico de la práctica 1
baseline_model = BaselineModel.load_model(MODELS_DIR / "baseline.pkl")

# Evaluamos el modelo baseline en el conjunto de evaluación
baseline_eval_predictions = baseline_model.predict(X_eval_baseline)
accuracy_eval_baseline = accuracy_score(y_eval_baseline, baseline_eval_predictions)
macrof1_eval_baseline = f1_score(y_eval_baseline, baseline_eval_predictions, average="macro")
perclassf1_eval_baseline = f1_score(y_eval_baseline, baseline_eval_predictions, average=None)
print("- Resultados del modelo baseline con el conjunto de evaluación:")
print(f"    - Accuracy: {accuracy_eval_baseline:.3f}")
print(f"    - Macro-F1: {macrof1_eval_baseline:.3f}")
print("    - Per-class F1:")
for i, label in ID2LABEL.items():
    print(f"        - {label}: {perclassf1_eval_baseline[i]:.3f}")

- Cargando modelo desde models/baseline.pkl...
- Resultados del modelo baseline con el conjunto de evaluación:
    - Accuracy: 0.650
    - Macro-F1: 0.648
    - Per-class F1:
        - rap: 0.841
        - rock: 0.579
        - pop: 0.525
        - other: 0.648


/home/wrstdani/dev/pln2_practica2/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SGDClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# Comparamos ambos clasificadores con el conjunto de test
baseline_test_predictions = baseline_model.predict(X_test_baseline)
accuracy_test_baseline = accuracy_score(y_test_baseline, baseline_test_predictions)
macrof1_test_baseline = f1_score(y_test_baseline, baseline_test_predictions, average="macro")
perclassf1_test_baseline = f1_score(y_test_baseline, baseline_test_predictions, average=None)

PRED_TEST["baseline_tfidf"] = baseline_test_predictions
METRICS_TEST["baseline_tfidf"] = {
    "accuracy": accuracy_test_baseline,
    "macrof1": macrof1_test_baseline,
    "perclassf1": perclassf1_test_baseline
}


print(f"- Accuracy: Baseline = {accuracy_test_baseline:.3f}; k-NN = {accuracy_test_knn:.3f}")
print(f"- Macro-F1: Baseline = {macrof1_test_baseline:.3f}; k-NN = {macrof1_test_knn:.3f}")
print("- F1 por clase:")
for i, label in ID2LABEL.items():
    print(f"        - {label}: Baseline={perclassf1_test_baseline[i]:.3f}, k-NN={perclassf1_test_knn[i]:.3f}")

- Accuracy: Baseline = 0.648; k-NN = 0.806
- Macro-F1: Baseline = 0.646; k-NN = 0.806
- F1 por clase:
        - rap: Baseline=0.840, k-NN=0.950
        - rock: Baseline=0.580, k-NN=0.721
        - pop: Baseline=0.524, k-NN=0.764
        - other: Baseline=0.642, k-NN=0.790


## 3. Clasificador híbrido Transformer + k-NN (RAG para Clasificación)
### 3.1. Implementación del clasificador híbrido
- Implementamos el clasificador híbrido con:
    - Varios valores de $\alpha \in \{0.0, 0.25, 0.5, 0.75, 1.0\}$.
    - $k$ fijo (por ejemplo, $k=5$).

In [ ]:
class HybridClassifier(KNNClassifier):
    def __init__(
            self,
            transformer_model: TransformerModel,
            device: torch.device,
            k: int,
            alpha: float
        ):
        """
        Constructor de la clase HybridClassifier

        :param transformer_model: Transformer para obtener embeddings y realizar inferencia.
        :type transformer_model: TransformerModel
        :param device: Dispositivo para PyTorch.
        :type device: torch.device
        :param k: Número de vecinos a considerar.
        :type k: int
        :param alpha: Peso del modelo k-NN en la combinación.
        :type alpha: float
        """
        super().__init__(transformer_model, device, k)
        self.alpha = alpha

    def fit(
            self,
            texts: dict | Dataset,
            train_embeddings: np.ndarray | None = None
        ):
        """
        Entrena el clasificador híbrido con el conjunto de entrenamiento dado.
        :param texts: Conjunto de entrenamiento.
        :type texts: dict | Dataset
        """
        self.train_dataset = texts
        self.train_embeddings = train_embeddings if train_embeddings is not None else self._embed_fn(texts)
        self.nearest_neighbors = NearestNeighbors(n_neighbors=self.k)
        self.nearest_neighbors.fit(self.train_embeddings)

    def predict(
            self,
            texts: dict | Dataset,
            embeddings: np.ndarray | None = None
        ) -> tuple[np.ndarray, np.ndarray]:
        """
        Realiza la predicción combinando k-NN y el modelo Transformer.
        :param texts: Conjunto de datos para la predicción.
        :type texts: dict | Dataset
        :param embeddings: Embeddings precomputados para la predicción.
        :type embeddings: np.ndarray | None
        :return: Índices de los vecinos y etiquetas predichas.
        :rtype: tuple[np.ndarray, np.ndarray]
        """
        _, knn_indices, _, _ = self._search(embeddings if embeddings is not None else self._embed_fn(texts))
        transformer_probs = self.__transformer_inference(texts=texts, embeddings=embeddings)
        predictions = []
        for i in range(len(texts) if texts is not None else len(embeddings)):
            # Calcular probabilidades del k-NN (voto mayoritario)
            probs_knn = np.zeros(len(self.transformer_model.label2id))
            for neighbor_idx in knn_indices[i]:
                neighbor_label = self.train_dataset[neighbor_idx]["label"]
                probs_knn[neighbor_label] += 1
            probs_knn /= self.k
            
            # Obtener probabilidades del Transformer (ya están como array numpy)
            probs_transformer = transformer_probs[i]

            # Combinar probabilidades con alpha
            combined_probs = self.alpha * probs_knn + (1 - self.alpha) * probs_transformer
            pred_label = np.argmax(combined_probs)
            predictions.append(pred_label)

        return knn_indices, np.array(predictions)
    
    def __transformer_inference(self, texts: dict | Dataset | None = None, embeddings: np.ndarray | None = None):
        """
        Realiza la inferencia con el modelo Transformer para los textos dados.
        Devuelve las probabilidades normalizadas (softmax) para clasificación multi-clase.
        
        :param texts: Textos para la inferencia.
        :type texts: dict | Dataset | None
        :param embeddings: Embeddings de los textos. Si se proporciona pero no textos, retorna distribución uniforme.
        :type embeddings: np.ndarray | None
        :return: Array (N, num_clases) con probabilidades para cada muestra y clase.
        :rtype: np.ndarray
        """
        # Si tenemos textos, hacer la inferencia del transformer
        if texts is not None:
            probs_list = []
            for text_id in tqdm.tqdm(range(len(texts)), desc="Inferencia con Transformer"):
                input_ids = torch.tensor(texts[text_id]["input_ids"]).to(self.device)
                attention_mask = torch.tensor(texts[text_id]["attention_mask"]).to(self.device)

                inputs = {"input_ids": input_ids.unsqueeze(0), "attention_mask": attention_mask.unsqueeze(0)}

                with torch.no_grad():
                    outputs = self.transformer_model._model(**inputs)
                    logits = outputs.logits.squeeze(0)
                    probs = torch.softmax(logits, dim=-1).cpu().numpy()
                
                probs_list.append(probs)
            
            return np.array(probs_list)
        
        # Si solo tenemos embeddings, retornar distribución uniforme
        elif embeddings is not None:
            num_classes = len(self.transformer_model.label2id)
            uniform_probs = np.ones((len(embeddings), num_classes)) / num_classes
            return uniform_probs
        
        else:
            raise ValueError("Se debe proporcionar textos o embeddings para la inferencia.")

In [ ]:
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
hybrid_classifiers = {}
for alpha in ALPHAS:
    hybrid_classifiers[str(alpha)] = HybridClassifier(
            transformer_model=teacher_model,
            device=DEVICE,
            k=N_NEIGHBORS,
            alpha=alpha
        )

### 3.2. Evaluación del clasificador híbrido
- Medimos el rendimiento del clasificador con *accuracy* y *macro-F1* en la partición de test para cada valor de $\alpha$.

In [ ]:
hybrid_eval_metrics = {}


for alpha_str, hybrid_classifier in hybrid_classifiers.items():
    hybrid_classifier.fit(train_dataset_transformer, train_embeddings)
    knn_indices, hybrid_predictions = hybrid_classifier.predict(eval_dataset_transformer, eval_embeddings)


    accuracy_eval_hybrid = accuracy_score(y_eval_baseline, hybrid_predictions)
    macrof1_eval_hybrid = f1_score(y_eval_baseline, hybrid_predictions, average="macro")
    perclassf1_eval_hybrid = f1_score(y_eval_baseline, hybrid_predictions, average=None)

    print(f"- Modelo híbrido (alpha={alpha_str}):")
    print(f"    - Accuracy: {accuracy_eval_hybrid:.3f}")
    print(f"    - Macro-F1: {macrof1_eval_hybrid:.3f}")
    print("    - F1 por clase:")
    for i, label in ID2LABEL.items():
        print(f"        - {label}: {perclassf1_eval_hybrid[i]:.3f}")
    print()

    hybrid_eval_metrics[alpha_str] = {
        "accuracy": accuracy_eval_hybrid,
        "macrof1": macrof1_eval_hybrid,
        "perclassf1": perclassf1_eval_hybrid
    }

Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 230.08it/s]


- Modelo híbrido (alpha=0.0):
    - Accuracy: 0.800
    - Macro-F1: 0.801
    - F1 por clase:
        - rap: 0.949
        - rock: 0.714
        - pop: 0.753
        - other: 0.788



Inferencia con Transformer: 100%|███████| 47638/47638 [03:26<00:00, 230.21it/s]


- Modelo híbrido (alpha=0.25):
    - Accuracy: 0.804
    - Macro-F1: 0.805
    - F1 por clase:
        - rap: 0.950
        - rock: 0.720
        - pop: 0.758
        - other: 0.792



Inferencia con Transformer: 100%|███████| 47638/47638 [03:26<00:00, 230.39it/s]


- Modelo híbrido (alpha=0.5):
    - Accuracy: 0.805
    - Macro-F1: 0.806
    - F1 por clase:
        - rap: 0.950
        - rock: 0.721
        - pop: 0.761
        - other: 0.792



Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 230.05it/s]


- Modelo híbrido (alpha=0.75):
    - Accuracy: 0.804
    - Macro-F1: 0.804
    - F1 por clase:
        - rap: 0.950
        - rock: 0.716
        - pop: 0.760
        - other: 0.791



Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 230.07it/s]


- Modelo híbrido (alpha=1.0):
    - Accuracy: 0.802
    - Macro-F1: 0.802
    - F1 por clase:
        - rap: 0.948
        - rock: 0.714
        - pop: 0.756
        - other: 0.790



- Comparamos los resultados con:
    - El modelo Transformer de la práctica 1 (equivalente a $\alpha=1$).
    - El clasificador k-NN puro de la sección 2 (equivalente a $\alpha=0$).

In [ ]:
# Realizamos las predicciones con el modelo teacher (Transformer)
teacher_test_predictions = teacher_model.predict(test_dataset_transformer)

- Realizando predicciones...


In [ ]:
# Evaluamos el modelo teacher en el conjunto de test
accuracy_test_teacher = accuracy_score(y_test_baseline, teacher_test_predictions)
macrof1_test_teacher = f1_score(y_test_baseline, teacher_test_predictions, average="macro")
perclassf1_test_teacher = f1_score(y_test_baseline, teacher_test_predictions, average=None)

PRED_TEST["teacher_transformer"] = teacher_test_predictions
METRICS_TEST["teacher_transformer"] = {
    "accuracy": accuracy_test_teacher,
    "macrof1": macrof1_test_teacher,
    "perclassf1": perclassf1_test_teacher
}


# Evaluamos los modelos híbridos en el conjunto de test
PRED_TEST["hybrid"] = {}
METRICS_TEST["hybrid"] = {}

for alpha_str, hybrid_classifier in hybrid_classifiers.items():
    knn_indices, hybrid_test_predictions = hybrid_classifier.predict(test_dataset_transformer, test_embeddings)
 
   
    accuracy_test_hybrid = accuracy_score(y_test_baseline, hybrid_test_predictions)
    macrof1_test_hybrid = f1_score(y_test_baseline, hybrid_test_predictions, average="macro")
    perclassf1_test_hybrid = f1_score(y_test_baseline, hybrid_test_predictions, average=None)

    PRED_TEST["hybrid"][alpha_str] = hybrid_test_predictions
    METRICS_TEST["hybrid"][alpha_str] = {
        "accuracy": accuracy_test_hybrid,
        "macrof1": macrof1_test_hybrid,
        "perclassf1": perclassf1_test_hybrid
    }
    
    print(f"- Clasificador híbrido (alpha={alpha_str}) en test:")
    print(f"    - Accuracy: {accuracy_test_hybrid:.3f}")
    print(f"    - Macro-F1: {macrof1_test_hybrid:.3f}")
    print("    - F1 por clase:")
    for i, label in ID2LABEL.items():
        print(f"        - {label}: {perclassf1_test_hybrid[i]:.3f}")
    print()

    

Inferencia con Transformer: 100%|███████| 47638/47638 [03:26<00:00, 230.19it/s]


- Clasificador híbrido (alpha=0.0) en test:
    - Accuracy: 0.803
    - Macro-F1: 0.804
    - F1 por clase:
        - rap: 0.951
        - rock: 0.717
        - pop: 0.762
        - other: 0.787



Inferencia con Transformer: 100%|███████| 47638/47638 [03:26<00:00, 230.17it/s]


- Clasificador híbrido (alpha=0.25) en test:
    - Accuracy: 0.806
    - Macro-F1: 0.807
    - F1 por clase:
        - rap: 0.951
        - rock: 0.722
        - pop: 0.765
        - other: 0.790



Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 230.05it/s]


- Clasificador híbrido (alpha=0.5) en test:
    - Accuracy: 0.808
    - Macro-F1: 0.809
    - F1 por clase:
        - rap: 0.951
        - rock: 0.725
        - pop: 0.768
        - other: 0.791



Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 229.19it/s]


- Clasificador híbrido (alpha=0.75) en test:
    - Accuracy: 0.807
    - Macro-F1: 0.808
    - F1 por clase:
        - rap: 0.951
        - rock: 0.724
        - pop: 0.767
        - other: 0.790



Inferencia con Transformer: 100%|███████| 47638/47638 [03:27<00:00, 229.97it/s]


- Clasificador híbrido (alpha=1.0) en test:
    - Accuracy: 0.806
    - Macro-F1: 0.806
    - F1 por clase:
        - rap: 0.950
        - rock: 0.721
        - pop: 0.764
        - other: 0.790



In [ ]:
# Comparamos los resultados del modelo híbrido con los resultados del modelo Transformer en el conjunto de test
for alpha_str, metrics in METRICS_TEST["hybrid"].items():
    print(f"- Con alfa = {alpha_str}:")
    print(f"    - Accuracy --> Clasificador híbrido = {metrics['accuracy']:.3f}; Teacher = {accuracy_test_teacher:.3f}")
    print(f"    - Macro-F1 --> Clasificador híbrido = {metrics['macrof1']:.3f}; Teacher = {macrof1_test_teacher:.3f}")
    print("    - Per-class F1:")
    for i, label in ID2LABEL.items():
        print(f"        - {label} --> Clasificador híbrido = {metrics['perclassf1'][i]:.3f}; Teacher = {perclassf1_test_teacher[i]:.3f}")
    print()

- Con alfa = 0.0:
    - Accuracy --> Clasificador híbrido = 0.803; Teacher = 0.803
    - Macro-F1 --> Clasificador híbrido = 0.804; Teacher = 0.804
    - Per-class F1:
        - rap --> Clasificador híbrido = 0.951; Teacher = 0.951
        - rock --> Clasificador híbrido = 0.717; Teacher = 0.717
        - pop --> Clasificador híbrido = 0.762; Teacher = 0.762
        - other --> Clasificador híbrido = 0.787; Teacher = 0.787

- Con alfa = 0.25:
    - Accuracy --> Clasificador híbrido = 0.806; Teacher = 0.803
    - Macro-F1 --> Clasificador híbrido = 0.807; Teacher = 0.804
    - Per-class F1:
        - rap --> Clasificador híbrido = 0.951; Teacher = 0.951
        - rock --> Clasificador híbrido = 0.722; Teacher = 0.717
        - pop --> Clasificador híbrido = 0.765; Teacher = 0.762
        - other --> Clasificador híbrido = 0.790; Teacher = 0.787

- Con alfa = 0.5:
    - Accuracy --> Clasificador híbrido = 0.808; Teacher = 0.803
    - Macro-F1 --> Clasificador híbrido = 0.809; Teacher = 0

## 4. Explicabilidad basada en retrieval
### 4.1. Explicaciones para ejemplos bien clasificados

In [ ]:
# Obtenemos 10 ejemplos bien clasificados
correct_indices = np.where(knn_test_predictions == y_test_baseline)[0]
correct_sample_indices = np.random.choice(correct_indices, size=min(10, len(correct_indices)), replace=False)

print(f"- Total de ejemplos bien clasificados: {len(correct_indices)}/{len(y_test_baseline)}")
print(f"- Mostrando {min(10, len(correct_indices))} ejemplos:\n")

for i, idx in enumerate(correct_sample_indices):
    example = test_dataset_transformer[idx]
    true_label = y_test_baseline[idx]
    pred_label = knn_test_predictions[idx]
    
    print(f"Ejemplo {i+1} (índice {idx}):")
    print(f"    - Texto: {example['text'][:150]}...")
    print(f"    - Etiqueta verdadera: {true_label}")
    print(f"    - Predicción: {pred_label}")
    print()

- Total de ejemplos bien clasificados: 38392/47638
- Mostrando 10 ejemplos:

Ejemplo 1 (índice 17533):
    - Texto: it's the bad guys
say hello to the bad guys 
big cat records gucci black magic

 
we the bad guys and we play for the bad team
we hauntin ya life like...
    - Etiqueta verdadera: 0
    - Predicción: 0

Ejemplo 2 (índice 23908):
    - Texto: bitch im young money x17
bitch im young

 
who the hottest niggas out boy young money
we can buy you niggas out if really wanted
car transformin' the ...
    - Etiqueta verdadera: 0
    - Predicción: 0

Ejemplo 3 (índice 29094):
    - Texto: thirstin howl iii the polorican 
 
 
 
 

 
yo this is for the live puerto rocks with official style thoroughbred status yo what s my nationality 

 
...
    - Etiqueta verdadera: 0
    - Predicción: 0

Ejemplo 4 (índice 9383):
    - Texto: oh na na what's my name 
oh na na what's my name 

oh na na what's my name 
oh na na what's my name 
oh na na what's my name 
what's my name what's my...
    -

#### 4.1.1. Comentario

### 4.2. Explicaciones para ejemplos mal clasificados

In [ ]:
# Obtenemos 10 ejemplos mal clasificados
incorrect_indices = np.where(knn_test_predictions != y_test_baseline)[0]
incorrect_sample_indices = np.random.choice(incorrect_indices, size=min(10, len(incorrect_indices)), replace=False)

print(f"- Total de ejemplos mal clasificados: {len(incorrect_indices)}/{len(y_test_baseline)}")
print(f"- Mostrando {min(10, len(incorrect_indices))} ejemplos:\n")
for i, idx in enumerate(incorrect_sample_indices):
    example = test_dataset_transformer[idx]
    true_label = y_test_baseline[idx]
    pred_label = knn_test_predictions[idx]
    
    print(f"Ejemplo {i+1} (índice {idx}):")
    print(f"    - Texto: {example['text'][:150]}...")
    print(f"    - Etiqueta verdadera: {true_label}")
    print(f"    - Predicción: {pred_label}")
    print()

- Total de ejemplos mal clasificados: 9246/47638
- Mostrando 10 ejemplos:

Ejemplo 1 (índice 11841):
    - Texto: bury all your secrets in my skin
come away with innocence and leave me with my sins
the air around me still feels like a cage
and love is just a camou...
    - Etiqueta verdadera: 1
    - Predicción: 2

Ejemplo 2 (índice 18042):
    - Texto: The road is narrow, the horizon wide And to say what's waiting on the other side Is so rewarding and the ultimate prize But what good is something if ...
    - Etiqueta verdadera: 1
    - Predicción: 3

Ejemplo 3 (índice 12351):
    - Texto: Haven't you heard it's getting that time 
So go spread the word and get practising your rhymes 
The foot and the bear and signature moves 
A word in y...
    - Etiqueta verdadera: 3
    - Predicción: 2

Ejemplo 4 (índice 9732):
    - Texto: My father tried to tell me son just get your feet on the ground
You're in deep way over your head just turn your life around
Poor mother, all I ever g...
    - E

#### 4.2.1. Comentario

## 5. Comprensión / modelo destilado
### 5.1. Elección del modelo destilado
- Seleccionamos un modelo más pequeño que el Transformer de la práctica 1. Como para éste se eligió `roberta-base`, optamos por `distilroberta-base`.

In [ ]:
student_model = TransformerModel(
    output_dir=MODELS_DIR,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    model_name="distilroberta-base"
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 5.2. Entrenamiento del modelo destilado
- Aplicamos *fine-tuning* del modelo destilado utilizando el mismo *dataset* que en la práctica 1.

In [ ]:
student_model.fit(train_dataset_transformer, eval_dataset_transformer)

- Iniciando entrenamiento del modelo distilroberta-base...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.634000,0.593605,0.748331,0.752135
2,0.468400,0.553445,0.773731,0.776939
3,0.447100,0.547880,0.793463,0.794845


Entrenamiento finalizado.


In [ ]:
# Evaluamos el modelo student en el conjunto de evaluación
student_eval_predictions = student_model.predict(eval_dataset_transformer)
accuracy_eval_student = accuracy_score(y_eval_baseline, student_eval_predictions)
macrof1_eval_student = f1_score(y_eval_baseline, student_eval_predictions, average="macro")
perclassf1_eval_student = f1_score(y_eval_baseline, student_eval_predictions, average=None)
print("- Resultados del modelo student con el conjunto de evaluación:")
print(f"    - Accuracy: {accuracy_eval_student:.3f}")
print(f"    - Macro-F1: {macrof1_eval_student:.3f}")
print("    - Per-class F1:")
for i, label in ID2LABEL.items():
    print(f"        - {label}: {perclassf1_eval_student[i]:.3f}")

- Realizando predicciones...


- Resultados del modelo student con el conjunto de evaluación:
    - Accuracy: 0.793
    - Macro-F1: 0.795
    - Per-class F1:
        - rap: 0.948
        - rock: 0.707
        - pop: 0.743
        - other: 0.782


## 5.3. Evaluación del modelo destilado
- Utilizamos las siguientes métricas de **calidad** con el conjunto de *test*:
    - *Accuracy*.
    - *Macro-F1*.
    - *Per-class F1*.
- Evaluamos también el **coste** (número aproximado de parámetros, tamaño del modelo en MB en disco y tiempo de inferencia por *batch*).

In [ ]:
# Realizamos predicciones con el modelo destilado (student)
student_test_predictions = student_model.predict(test_dataset_transformer)

- Realizando predicciones...


In [ ]:
# Métricas de calidad
accuracy_test_student = accuracy_score(y_test_baseline, student_test_predictions)
macrof1_test_student = f1_score(y_test_baseline, student_test_predictions, average="macro")
perclassf1_test_student = f1_score(y_test_baseline, student_test_predictions, average=None)

PRED_TEST["student_distilled"] = student_test_predictions
METRICS_TEST["student_distilled"] = {
    "accuracy": accuracy_test_student,
    "macrof1": macrof1_test_student,
    "perclassf1": perclassf1_test_student
}


print(f"- Accuracy: {accuracy_test_student:.3f}")
print(f"- Macro-F1: {macrof1_test_student:.3f}")
print(f"- Per-Class F1:")
for i, label in ID2LABEL.items():
    print(f"    - {label} = {perclassf1_test_student[i]}")

- Accuracy: 0.796
- Macro-F1: 0.797
- Per-Class F1:
    - rap = 0.9494685338282436
    - rock = 0.7087747035573122
    - pop = 0.7491037914891823
    - other = 0.7808662499445849


In [ ]:
student_model.save_model(MODELS_DIR / "student_transformer")

- Guardando modelo en models/student_transformer...


In [ ]:
# Calcular tamaño de la carpeta que contiene el modelo student
def get_folder_size_mb(folder_path):
    total = 0
    for item in Path(folder_path).rglob('*'):
        if item.is_file():
            total += item.stat().st_size
    return total / (1024 * 1024)

student_size_mb = get_folder_size_mb(MODELS_DIR / "student_transformer")
print(f"- Tamaño del modelo student en disco: {student_size_mb:.2f} MB")

- Tamaño del modelo student en disco: 313.29 MB


### 5.4. Análisis de resultados
- Analizamos la calidad que se pierde al pasar del modelo *teacher* (Transformer de la práctica 1) al modelo *student* (modelo destilado).

In [ ]:
print(f"- Accuracy --> Teacher = {accuracy_test_teacher:.3f}; Student = {accuracy_test_student:.3f}")
print(f"- Macro-F1 --> Teacher = {macrof1_test_teacher:.3f}; Student = {macrof1_test_student:.3f}")
print("- Per-class F1:")
for i, label in ID2LABEL.items():
    print(f"    - {label} --> Teacher = {perclassf1_test_teacher[i]:.3f}; Student = {perclassf1_test_student[i]:.3f}")

- Accuracy --> Teacher = 0.803; Student = 0.796
- Macro-F1 --> Teacher = 0.804; Student = 0.797
- Per-class F1:
    - rap --> Teacher = 0.951; Student = 0.949
    - rock --> Teacher = 0.717; Student = 0.709
    - pop --> Teacher = 0.762; Student = 0.749
    - other --> Teacher = 0.787; Student = 0.781


- Observamos la diferencia de velocidad de inferencia entre ambos modelos.

In [ ]:
# Inferencia con el modelo teacher
start = time.time()
teacher_test_predictions = teacher_model.predict(test_dataset_transformer)
end = time.time()
print(f"- Tiempo de inferencia del modelo teacher en el conjunto de test: {end - start:.3f} segundos")

- Realizando predicciones...


- Tiempo de inferencia del modelo teacher en el conjunto de test: 92.244 segundos


In [ ]:
# Inferencia con el modelo student
start = time.time()
student_test_predictions = student_model.predict(test_dataset_transformer)
end = time.time()
print(f"- Tiempo de inferencia del modelo student en el conjunto de test: {end - start:.3f} segundos")

- Realizando predicciones...


- Tiempo de inferencia del modelo student en el conjunto de test: 41.528 segundos


- Visualizar si el modelo destilado se degrada más en ciertas clases.

## 6. Resumen automático como explicación
### 6.1. Resumen global por clase
- Para cada clase, seleccionamos aproximadamente 50 textos de entrenamiento, concatenamos fragmentos en un "pseudo-documento" y usamos un modelo generativo (p. ej., T5-small o BART-base) para generar un resumen de 3–5 frases que describa la clase.

In [ ]:
# Cargamos T5-small como modelo generativo
tokenizer = AutoTokenizer.from_pretrained("t5-small")
generative_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(DEVICE)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### 6.2. Resumen local basado en vecinos (explicación de predicciones)
- Seleccionar al menos 6 ejemplos de test (3 correctos, 3 erróneos). Para cada uno, utilizamos el módulo de *retrieval* para obtener los $k$ vecinos más cercanos al conjunto de entrenamiento.

## 7. Análisis global y discusión
### 7.1. Comparación global
- Comparar los baselines de P1, k-NN, el híbrido y el modelo comprimido, analizando el equilibrio entre calidad, coste e interpretabilidad.

In [ ]:
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
LABEL_NAMES = [ID2LABEL[i] for i in sorted(ID2LABEL.keys())]

best_alpha = None
if "hybrid" in METRICS_TEST and len(METRICS_TEST["hybrid"]) > 0:
    best_alpha = max(
        METRICS_TEST["hybrid"].items(),
        key=lambda kv: kv[1]["macrof1"]
    )[0]

rows = []

for name, m in METRICS_TEST.items():
    if name == "hybrid":
        continue
    row = {
        "model": name,
        "accuracy": m["accuracy"],
        "macro_f1": m["macrof1"],
    }
    for i, label in enumerate(LABEL_NAMES):
        row[f"f1_{label}"] = float(m["perclassf1"][i])
    rows.append(row)

if best_alpha is not None:
    m = METRICS_TEST["hybrid"][best_alpha]
    row = {
        "model": f"hybrid_best_alpha_{best_alpha}",
        "accuracy": m["accuracy"],
        "macro_f1": m["macrof1"],
    }
    for i, label in enumerate(LABEL_NAMES):
        row[f"f1_{label}"] = float(m["perclassf1"][i])
    rows.append(row)

rows_sorted = sorted(rows, key=lambda r: r["macro_f1"], reverse=True)

for r in rows_sorted:
    print(
        f"{r['model']:25s} | "
        f"Acc={r['accuracy']:.3f} | "
        f"Macro-F1={r['macro_f1']:.3f}"
    )


In [ ]:
models = [r["model"] for r in rows_sorted]
macro_f1s = [r["macro_f1"] for r in rows_sorted]

plt.figure(figsize=(10, 5))
plt.bar(models, macro_f1s)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Macro-F1")
plt.title("Comparación global (test)")
plt.tight_layout()
plt.show()

plt.savefig(FIGURES_DIR / "macro_f1_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### 7.2. Sesgo y ética
- Analizar aspectos de equidad y sesgo, como si hay clases peor tratadas o si los vecinos podrían agravar sesgos.

In [ ]:
def recall_per_class(y_true, y_pred, nC):
    return recall_score(y_true, y_pred, average=None, labels=list(range(nC)), zero_division=0)

# modelos principales
preds_bias = {
    "baseline_tfidf": PRED_TEST["baseline_tfidf"],
    "knn_embeddings": PRED_TEST["knn_embeddings"],
    "teacher_transformer": PRED_TEST["teacher_transformer"],
    "student_distilled": PRED_TEST["student_distilled"],
}

# híbrido best alpha
if best_alpha is not None:
    preds_bias[f"hybrid_best_{best_alpha}"] = PRED_TEST["hybrid"][best_alpha]

# plot recall
models = list(preds_bias.keys())
nC = len(LABEL_NAMES)
mat = np.array([recall_per_class(y_test_baseline, preds_bias[m], nC) for m in models])

x = np.arange(nC)
width = 0.85 / len(models)
plt.figure(figsize=(12, 5))
for i, m in enumerate(models):
    plt.bar(x + i*width, mat[i], width=width, label=m)
plt.xticks(x + (len(models)-1)*width/2, LABEL_NAMES)
plt.ylim(0, 1)
plt.ylabel("RECALL")
plt.title("Recall por clase (test) — sesgo/clases peor tratadas")
plt.legend(fontsize=8)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "recall_per_class.png", dpi=300, bbox_inches="tight")
plt.show()

### 7.3. Utilidad de la explicabilidad y riesgos
- Reflexionar sobre la utilidad real de las explicaciones basadas en tokens/atributos (P1), ejemplos (*retrieval*) y explicaciones generativas (resumen).